In [27]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from scipy.stats import spearmanr
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [28]:
# train = pd.read_csv("/kaggle/input/competitions/mitsui-commodity-prediction-challenge/train.csv")
# test = pd.read_csv('/kaggle/input/competitions/mitsui-commodity-prediction-challenge/test.csv')
# target_pairs = pd.read_csv('/kaggle/input/competitions/mitsui-commodity-prediction-challenge/target_pairs.csv')
# train_labels = pd.read_csv('/kaggle/input/competitions/mitsui-commodity-prediction-challenge/train_labels.csv')

In [29]:
train = pd.read_csv('/Users/hayden/coderepos_mac_mini/mitsui_commodity/data/train.csv')
test = pd.read_csv('/Users/hayden/coderepos_mac_mini/mitsui_commodity/data/test.csv')
train_labels = pd.read_csv('/Users/hayden/coderepos_mac_mini/mitsui_commodity/data/train_labels.csv')
target_pairs = pd.read_csv('/Users/hayden/coderepos_mac_mini/mitsui_commodity/data/target_pairs.csv')

In [30]:
feature_cols = [c for c in train.columns if c != 'date_id']
target_cols = [c for c in train_labels.columns if c != 'date_id']


In [31]:
#not really sure aabout this part
df = train.merge(train_labels, on='date_id', how='inner').sort_values('date_id').reset_index(drop=True)

In [32]:
#might replace this with just -1 instead
df[feature_cols] = df[feature_cols].ffill().bfill().fillna(0)
df[target_cols] = df[target_cols].ffill().bfill().fillna(0)

In [33]:
#scale features
scaler = StandardScaler()
X_full = scaler.fit_transform(df[feature_cols]).astype(np.float32)
y_full = scaler.fit_transform(df[target_cols]).astype(np.float32)

In [34]:
#sliding windows
SEQ_LEN = 30
def make_windows(X, y, seq_len=SEQ_LEN):
    Xs, ys = [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i:i+seq_len])
        ys.append(y[i+seq_len])
    return np.array(Xs), np.array(ys)

X_seq, y_seq = make_windows(X_full, y_full, SEQ_LEN)


In [35]:
#time based train/val split
split = int(.85 * len(X_seq))
X_tr, X_val = X_seq[:split], X_seq[split:]
y_tr, y_val = y_seq[:split], y_seq[split:]

BATCH_SIZE = 64

class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = torch.from_numpy(X), torch.from_numpy(y)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(SeqDataset(X_tr, y_tr), batch_size = BATCH_SIZE, shuffle=True)
val_loader = DataLoader(SeqDataset(X_val, y_val), batch_size = BATCH_SIZE, shuffle=False)

In [36]:
class LSTMRegressor(nn.Module):
    def __init__(self, n_features, n_targets, hidden=128, layers=2, dropout=.2):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, num_layers=layers, dropout=dropout, batch_first=True)
        self.head= nn.Linear(hidden, n_targets)
    def forward(self, X):
        out, _ = self.lstm(X)
        return self.head(out[:, -1, :])  # take last time step
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LSTMRegressor(len(feature_cols), len(target_cols)).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

In [37]:
#train loop 
for epoch in range(150):
    model.train()
    tr_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        opt.step()
        tr_loss += loss.item()
    model.eval()
    v_loss = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            v_loss += loss_fn(model(xb), yb).item()
    print(f'Epoch {epoch+1} - Train Loss: {tr_loss/len(train_loader):.4f} - Val Loss: {v_loss/len(val_loader):.4f}')

Epoch 1 - Train Loss: 1.0362 - Val Loss: 0.8021
Epoch 2 - Train Loss: 1.0236 - Val Loss: 0.8033
Epoch 3 - Train Loss: 1.0106 - Val Loss: 0.8158
Epoch 4 - Train Loss: 0.9871 - Val Loss: 0.8122
Epoch 5 - Train Loss: 0.9676 - Val Loss: 0.8208
Epoch 6 - Train Loss: 0.9494 - Val Loss: 0.8684
Epoch 7 - Train Loss: 0.9326 - Val Loss: 0.8629
Epoch 8 - Train Loss: 0.9328 - Val Loss: 0.8548
Epoch 9 - Train Loss: 0.9008 - Val Loss: 0.8679
Epoch 10 - Train Loss: 0.8739 - Val Loss: 0.8671
Epoch 11 - Train Loss: 0.8598 - Val Loss: 0.9009
Epoch 12 - Train Loss: 0.8496 - Val Loss: 0.9314
Epoch 13 - Train Loss: 0.8286 - Val Loss: 0.9809
Epoch 14 - Train Loss: 0.8105 - Val Loss: 0.9364
Epoch 15 - Train Loss: 0.8086 - Val Loss: 0.9160
Epoch 16 - Train Loss: 0.7886 - Val Loss: 0.8833
Epoch 17 - Train Loss: 0.7662 - Val Loss: 0.9105
Epoch 18 - Train Loss: 0.7546 - Val Loss: 0.9101
Epoch 19 - Train Loss: 0.7399 - Val Loss: 0.8670
Epoch 20 - Train Loss: 0.7253 - Val Loss: 0.8519
Epoch 21 - Train Loss: 0.7188